In [17]:
from neo4j import GraphDatabase
import json
from datetime import date, datetime

# --- Connection details ---
uri = "neo4j://107.21.198.63:7687"  # or "neo4j+s://" if SSL needed
user = "neo4j"
password = "Ascentt-GraphDB"

driver = GraphDatabase.driver(uri, auth=(user, password))

# --- Custom JSON Encoder for Neo4j date/time objects ---
class Neo4jJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (datetime, date)):
            return obj.isoformat()
        try:
            # Fallback to string for any unknown Neo4j types
            return str(obj)
        except Exception:
            return super().default(obj)

def export_data():
    with driver.session() as session:
        print("✅ Connected to Neo4j successfully!")

        # Fetch all nodes
        print("📦 Fetching all nodes...")
        node_query = "MATCH (n) RETURN elementId(n) AS id, labels(n) AS labels, properties(n) AS props"
        nodes = [record.data() for record in session.run(node_query)]

        # Fetch all relationships
        print("🔗 Fetching all relationships...")
        rel_query = """
        MATCH (a)-[r]->(b)
        RETURN 
            elementId(a) AS start_id,
            elementId(b) AS end_id,
            type(r) AS type,
            properties(r) AS props
        """
        relationships = [record.data() for record in session.run(rel_query)]

        # Combine and save
        graph_data = {
            "nodes": nodes,
            "relationships": relationships
        }

        with open("neo4j_full_export.json", "w") as f:
            json.dump(graph_data, f, indent=2, cls=Neo4jJSONEncoder)

        print("✅ Export complete! Data saved as neo4j_full_export.json")

if __name__ == "__main__":
    export_data()
    driver.close()


✅ Connected to Neo4j successfully!
📦 Fetching all nodes...
🔗 Fetching all relationships...
✅ Export complete! Data saved as neo4j_full_export.json


In [18]:
import json

# --- Step 1: Load the exported Neo4j JSON file ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"  # ✅ Fixed

# Open and load JSON into Python
with open(file_path, "r") as f:
    graph_data = json.load(f)

# Print high-level summary
print("✅ File loaded successfully!")
print(f"Total nodes: {len(graph_data['nodes'])}")
print(f"Total relationships: {len(graph_data['relationships'])}")

# Print first 2 nodes (sample)
print("\n📦 Sample Nodes:")
for node in graph_data["nodes"][:2]:
    print(json.dumps(node, indent=2))

# Print first 2 relationships (sample)
print("\n🔗 Sample Relationships:")
for rel in graph_data["relationships"][:2]:
    print(json.dumps(rel, indent=2))


✅ File loaded successfully!
Total nodes: 334
Total relationships: 1129

📦 Sample Nodes:
{
  "id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7812",
  "labels": [
    "SERIES"
  ],
  "props": {
    "Series": "COROLLA CROSS"
  }
}
{
  "id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7813",
  "labels": [
    "SERIES"
  ],
  "props": {
    "Series": "TACOMA"
  }
}

🔗 Sample Relationships:
{
  "start_id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8132",
  "end_id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7817",
  "type": "ASSOCIATED_SPEC200",
  "props": {}
}
{
  "start_id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8132",
  "end_id": "4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7818",
  "type": "ASSOCIATED_SPEC200",
  "props": {}
}


In [19]:
pip install networkx matplotlib

Note: you may need to restart the kernel to use updated packages.


In [20]:
import json
import networkx as nx

# --- Step 2: Convert JSON into a NetworkX graph with readable names ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"

# Load the JSON
with open(file_path, "r") as f:
    graph_data = json.load(f)

# Create a directed graph
G = nx.DiGraph()

# --- Add Nodes ---
for node in graph_data["nodes"]:
    node_id = node["id"]
    labels = node.get("labels", [])
    props = node.get("props", {})

    # Try to make a friendly name
    label_part = labels[0] if labels else "Node"
    key_values = []

    # Pick up to 2 interesting numeric/string properties
    for k, v in props.items():
        if isinstance(v, (str, int, float)) and k not in ["id", "uuid"]:
            key_values.append(str(v))
        if len(key_values) >= 2:
            break

    if key_values:
        display_name = f"{label_part}_{'_'.join(key_values)}"
    else:
        display_name = f"{label_part}_{node_id}"

    # Add node
    G.add_node(
        node_id,
        display_name=display_name,
        labels=labels,
        **props
    )

# --- Add Relationships (edges) ---
for rel in graph_data["relationships"]:
    start = rel["start_id"]
    end = rel["end_id"]
    rel_type = rel.get("type", "RELATIONSHIP")
    props = rel.get("props", {})

    G.add_edge(start, end, type=rel_type, **props)

# --- Summary ---
print("✅ Graph successfully created!")
print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")

print("\n📦 Sample Nodes:")
for n, data in list(G.nodes(data=True))[:3]:
    print(f"{data.get('display_name')} ({n}) → labels={data.get('labels')}")

print("\n🔗 Sample Relationships:")
for u, v, data in list(G.edges(data=True))[:3]:
    rel_name = data.get("type")
    print(f"{G.nodes[u].get('display_name')} → {G.nodes[v].get('display_name')} (type={rel_name})")


✅ Graph successfully created!
Total nodes: 334
Total edges: 1128

📦 Sample Nodes:
SERIES_COROLLA CROSS (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7812) → labels=['SERIES']
SERIES_TACOMA (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7813) → labels=['SERIES']
PLANTLINE_52 (APOLLO)_100 (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7814) → labels=['PLANTLINE']

🔗 Sample Relationships:
SERIES_COROLLA CROSS → MODEL_NUMBER_6303 (type=HAVING_MODEL_NUMBER)
SERIES_COROLLA CROSS → MODEL_NUMBER_6304 (type=HAVING_MODEL_NUMBER)
SERIES_COROLLA CROSS → MODEL_NUMBER_6305 (type=HAVING_MODEL_NUMBER)


In [21]:
import json
import networkx as nx

# --- Step 3: Explore the graph structure (readable version, no matplotlib) ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"

# Load data
with open(file_path, "r") as f:
    graph_data = json.load(f)

# --- Build a readable directed graph ---
G = nx.DiGraph()

for node in graph_data["nodes"]:
    node_id = node["id"]
    labels = node.get("labels", [])
    props = node.get("props", {})

    # Make a friendly display name
    label_part = labels[0] if labels else "Node"
    key_values = []
    for k, v in props.items():
        if isinstance(v, (str, int, float)) and k not in ["id", "uuid"]:
            key_values.append(str(v))
        if len(key_values) >= 2:
            break
    display_name = f"{label_part}_{'_'.join(key_values)}" if key_values else f"{label_part}_{node_id}"

    G.add_node(node_id, display_name=display_name, labels=labels, **props)

for rel in graph_data["relationships"]:
    start = rel["start_id"]
    end = rel["end_id"]
    rel_type = rel.get("type", "RELATIONSHIP")
    props = rel.get("props", {})
    G.add_edge(start, end, type=rel_type, **props)

# --- Summary ---
print("✅ Graph loaded successfully!")
print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")

# --- Basic Graph Statistics ---
print("\n📊 Basic Graph Statistics:")
print(f"Density: {nx.density(G):.4f}")
print(f"Is Directed: {G.is_directed()}")
print(f"Is Strongly Connected: {nx.is_strongly_connected(G)}")

# --- Degree Analysis ---
print("\n🔎 Node Degree Info:")
degree_dict = dict(G.degree())
sorted_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)

print("Top 5 most connected nodes:")
for node_id, deg in sorted_degree[:5]:
    name = G.nodes[node_id].get("display_name")
    print(f"  {name} ({node_id}): degree={deg}")

# --- Isolated Nodes (no connections) ---
isolated_nodes = list(nx.isolates(G))
print(f"\n🟡 Isolated Nodes (no connections): {len(isolated_nodes)}")
if isolated_nodes:
    print("Example:", [G.nodes[i].get("display_name") for i in isolated_nodes[:5]])

# --- Relationship Type Summary ---
print("\n🔗 Relationship Types:")
rel_types = {}
for _, _, data in G.edges(data=True):
    rel_type = data.get("type", "Unknown")
    rel_types[rel_type] = rel_types.get(rel_type, 0) + 1
for t, count in rel_types.items():
    print(f"  {t}: {count}")


✅ Graph loaded successfully!
Total nodes: 334
Total edges: 1128

📊 Basic Graph Statistics:
Density: 0.0101
Is Directed: True
Is Strongly Connected: False

🔎 Node Degree Info:
Top 5 most connected nodes:
  SERIES_TACOMA (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7813): degree=300
  SPEC_UUU35_053G (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8132): degree=156
  SPEC_UUU36_023J (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8133): degree=132
  SPEC_UUW40_010J (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8135): degree=111
  SPEC_UUU08_067B (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8127): degree=100

🟡 Isolated Nodes (no connections): 1
Example: ['_Neodash_Dashboard_{\n  "title": "GAINS DASHBOARD",\n  "version": "2.4",\n  "settings": {\n    "pagenumber": 1,\n    "editable": true,\n    "fullscreenEnabled": false,\n    "parameters": {\n      "neodash_series_series_type": "3W",\n      "neodash_series_series_type_display": "3W",\n      "neodash_namc_namc": "TMMI",\n      "neodash_namc_namc_display": "TMMI",\n      

In [22]:
import json
import networkx as nx

# --- Step 4: Analyze node influence and direction (readable version, no matplotlib) ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"

# Load data
with open(file_path, "r") as f:
    graph_data = json.load(f)

# --- Build a readable directed graph ---
G = nx.DiGraph()

for node in graph_data["nodes"]:
    node_id = node["id"]
    labels = node.get("labels", [])
    props = node.get("props", {})

    # Friendly display name
    label_part = labels[0] if labels else "Node"
    key_values = []
    for k, v in props.items():
        if isinstance(v, (str, int, float)) and k not in ["id", "uuid"]:
            key_values.append(str(v))
        if len(key_values) >= 2:
            break
    display_name = f"{label_part}_{'_'.join(key_values)}" if key_values else f"{label_part}_{node_id}"

    G.add_node(node_id, display_name=display_name, labels=labels, **props)

for rel in graph_data["relationships"]:
    start = rel["start_id"]
    end = rel["end_id"]
    rel_type = rel.get("type", "RELATIONSHIP")
    props = rel.get("props", {})
    G.add_edge(start, end, type=rel_type, **props)

print("✅ Graph loaded successfully!")
print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

# --- Degree Centrality ---
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())

print("\n📈 Top 5 nodes by OUT-degree (influencers / possible causes):")
for node, val in sorted(out_deg.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {G.nodes[node].get('display_name')} ({node}): out-degree = {val}")

print("\n📉 Top 5 nodes by IN-degree (receivers / possible effects):")
for node, val in sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {G.nodes[node].get('display_name')} ({node}): in-degree = {val}")

# --- PageRank (overall influence) ---
pagerank = nx.pagerank(G)
print("\n⭐ Top 5 nodes by PageRank (overall influence):")
for node, val in sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {G.nodes[node].get('display_name')} ({node}): PageRank = {val:.4f}")

# --- Betweenness Centrality (bridges / mediators) ---
bet = nx.betweenness_centrality(G)
print("\n🔗 Top 5 nodes by Betweenness Centrality (connectors / mediators):")
for node, val in sorted(bet.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {G.nodes[node].get('display_name')} ({node}): Betweenness = {val:.4f}")

# --- HITS Algorithm (Hubs & Authorities) ---
try:
    hubs, auths = nx.hits(G, max_iter=1000)
    print("\n💡 Top 5 Hubs (sources of influence):")
    for node, val in sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {G.nodes[node].get('display_name')} ({node}): Hub Score = {val:.4f}")

    print("\n📚 Top 5 Authorities (trusted targets):")
    for node, val in sorted(auths.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {G.nodes[node].get('display_name')} ({node}): Authority Score = {val:.4f}")
except nx.PowerIterationFailedConvergence:
    print("\n⚠️ HITS algorithm did not converge (graph too large or too sparse).")


✅ Graph loaded successfully!
Nodes: 334 | Edges: 1128

📈 Top 5 nodes by OUT-degree (influencers / possible causes):
  SERIES_TACOMA (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7813): out-degree = 292
  SPEC_UUU35_053G (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8132): out-degree = 155
  SPEC_UUU36_023J (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8133): out-degree = 131
  SPEC_UUW40_010J (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8135): out-degree = 110
  SPEC_UUU08_067B (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8127): out-degree = 98

📉 Top 5 nodes by IN-degree (receivers / possible effects):
  MODEL_YEAR_2024 (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8124): in-degree = 23
  PLANTLINE_60 (LINE 60)_100 (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7815): in-degree = 9
  NAMC_TMMBC (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:8126): in-degree = 9
  SERIES_TACOMA (4:02615720-87a9-4bcb-93d9-27fa09c36f9e:7813): in-degree = 8
  SPEC_200_30000_R  A DD  KEI B  C IBI JC  BBII XIBIFCGB EDB   B L BBG           I BI DB     FB 

In [23]:
import json
import pandas as pd
import networkx as nx
import numpy as np
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils

# --- Step 6A: Load data ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"
with open(file_path, "r") as f:
    graph_data = json.load(f)

# --- Build directed graph (with readable names) ---
G = nx.DiGraph()

for node in graph_data["nodes"]:
    node_id = node["id"]
    labels = node.get("labels", [])
    props = node.get("props", {})

    label_part = labels[0] if labels else "Node"
    key_values = []
    for k, v in props.items():
        if isinstance(v, (str, int, float)) and k not in ["id", "uuid"]:
            key_values.append(str(v))
        if len(key_values) >= 2:
            break
    display_name = f"{label_part}_{'_'.join(key_values)}" if key_values else f"{label_part}_{node_id}"

    G.add_node(node_id, display_name=display_name, labels=labels, **props)

for rel in graph_data["relationships"]:
    start = rel["start_id"]
    end = rel["end_id"]
    rel_type = rel.get("type", "RELATIONSHIP")
    props = rel.get("props", {})
    G.add_edge(start, end, type=rel_type, **props)

print("✅ Graph loaded for causal discovery")
print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

# --- Step 6B: Extract numeric features from node properties ---
numeric_nodes = []
numeric_data = []

for node_id, props in G.nodes(data=True):
    nums = {k: v for k, v in props.items() if isinstance(v, (int, float))}
    if nums:
        numeric_nodes.append(G.nodes[node_id].get("display_name"))
        numeric_data.append(nums)

if not numeric_data:
    raise ValueError("⚠️ No numeric features found in your graph nodes!")

df = pd.DataFrame(numeric_data).fillna(0)
df.index = numeric_nodes

print("\n📊 Sample numeric data:")
print(df.head())

# --- Step 6C: Data Cleaning to avoid singular matrix ---
print("\n🧹 Cleaning data for causal discovery...")

# 1. Drop constant columns
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
if const_cols:
    print(f"🗑️ Dropping constant columns: {const_cols}")
    df = df.drop(columns=const_cols)

# 2. Drop duplicate columns
df = df.T.drop_duplicates().T

# 3. Drop highly correlated columns (> 0.95)
corr = df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
if to_drop:
    print(f"🧾 Dropping highly correlated columns: {to_drop}")
    df = df.drop(columns=to_drop)

# 4. Add small noise if still singular or nearly singular
cond_number = np.linalg.cond(np.corrcoef(df.T))
print(f"Condition number: {cond_number:.2e}")
if cond_number > 1e12:
    print("⚠️ Adding small random noise to avoid singularity.")
    df += np.random.normal(0, 1e-6, df.shape)

# 5. Check samples vs variables
if df.shape[0] <= df.shape[1]:
    print(f"⚠️ Only {df.shape[0]} samples for {df.shape[1]} variables.")
    print("   Consider collecting more data or using dimensionality reduction (e.g., PCA).")

# --- Step 6D: Run PC Algorithm ---
print("\n🔍 Running PC Algorithm to infer causal structure...")

data = df.to_numpy()
variable_names = df.columns.tolist()

try:
    cg = pc(data, alpha=0.05, indep_test="fisherz", labels=variable_names)
    dot_graph = GraphUtils.to_pydot(cg.G).to_string()

    for i, col in enumerate(variable_names):
        dot_graph = dot_graph.replace(f"X{i+1}", col)

    print("\n✅ Causal Discovery Results (with real variable names):")
    print(dot_graph)

except ValueError as e:
    print(f"\n❌ PC Algorithm failed: {e}")
    print("Try reducing correlated features or ensuring more observations per variable.")


✅ Graph loaded for causal discovery
Nodes: 334 | Edges: 1128

📊 Sample numeric data:
                                                    Total_Capacity  \
PLANTLINE_52 (APOLLO)_100                                    100.0   
PLANTLINE_60 (LINE 60)_100                                   100.0   
SPEC_200_30000_R  A FD  LEI B  C IIB JC  BBCI X...             0.0   
SPEC_200_30000_R  A FD  JEI B  C IBB JC  BBII X...             0.0   
SPEC_200_30000_R  A FD  JEI B  C IBB JC  BBII X...             0.0   

                                                    Retail_Cost  Demand  \
PLANTLINE_52 (APOLLO)_100                                   0.0     0.0   
PLANTLINE_60 (LINE 60)_100                                  0.0     0.0   
SPEC_200_30000_R  A FD  LEI B  C IIB JC  BBCI X...      30000.0   870.0   
SPEC_200_30000_R  A FD  JEI B  C IBB JC  BBII X...      30000.0  1350.0   
SPEC_200_30000_R  A FD  JEI B  C IBB JC  BBII X...      30000.0  1280.0   

                                           























Depth=2, working on node 4: 100%|██████████| 5/5 [00:00<00:00, 607.22it/s]


✅ Causal Discovery Results (with real variable names):
digraph {
fontsize=18;
dpi=200;
0 [label=Total_Capacity];
0 [label=Total_Capacity];
1 [label=Retail_Cost];
1 [label=Retail_Cost];
2 [label=Demand];
2 [label=Demand];
3 [label=PPR1_Guide_Limit];
3 [label=PPR1_Guide_Limit];
4 [label=Demand_Gap];
4 [label=Demand_Gap];
0 -> 1 [dir=both, arrowtail=none, arrowhead=normal];
1 -> 2 [dir=both, arrowtail=none, arrowhead=normal];
3 -> 1 [dir=both, arrowtail=none, arrowhead=normal];
}



In [24]:
pip install econml

Note: you may need to restart the kernel to use updated packages.


In [25]:
# --- Step 7: Quantify causal effects using a Causal Forest ---
import json
import pandas as pd
import networkx as nx
import numpy as np
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# --- Load your Neo4j-exported graph ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"
with open(file_path, "r") as f:
    graph_data = json.load(f)

# --- Rebuild directed graph and extract numeric properties ---
G = nx.DiGraph()
for node in graph_data["nodes"]:
    G.add_node(node["id"], **node.get("props", {}))
for rel in graph_data["relationships"]:
    G.add_edge(rel["start_id"], rel["end_id"], type=rel["type"], **rel.get("props", {}))

# --- Extract numeric properties into DataFrame ---
numeric_nodes = []
numeric_data = []
for node_id, props in G.nodes(data=True):
    nums = {k: v for k, v in props.items() if isinstance(v, (int, float))}
    if nums:
        numeric_nodes.append(node_id)
        numeric_data.append(nums)

df = pd.DataFrame(numeric_data).fillna(0)
print("✅ Numeric dataset ready for causal analysis")
print(df.head())

# --- Step 7A: Clean numeric data to prevent singular matrix ---
print("\n🧹 Cleaning numeric data before causal discovery...")

# Drop constant columns
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
if const_cols:
    print(f"🗑️ Dropping constant columns: {const_cols}")
    df = df.drop(columns=const_cols)

# Drop duplicate columns
df = df.T.drop_duplicates().T

# Drop highly correlated columns (> 0.95)
corr = df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
if to_drop:
    print(f"🧾 Dropping highly correlated columns: {to_drop}")
    df = df.drop(columns=to_drop)

# Add small noise if condition number is large
cond_number = np.linalg.cond(np.corrcoef(df.T))
print(f"Condition number: {cond_number:.2e}")
if cond_number > 1e12:
    print("⚠️ Adding small random noise to avoid singularity.")
    df += np.random.normal(0, 1e-6, df.shape)

# Warn if fewer samples than variables
if df.shape[0] <= df.shape[1]:
    print(f"⚠️ Only {df.shape[0]} samples for {df.shape[1]} variables.")
    print("   Consider collecting more data or reducing features.")

# --- Step 7B: Run PC Algorithm ---
print("\n🔍 Running PC Algorithm to infer causal structure...")

data = df.to_numpy()
variable_names = df.columns.tolist()

try:
    cg = pc(data, alpha=0.05, indep_test="fisherz", labels=variable_names)

    # Extract edges
    edges = []
    for i in range(len(cg.G.graph)):
        for j in range(len(cg.G.graph)):
            if cg.G.graph[i][j] != 0:  # nonzero = edge
                edges.append((variable_names[i], variable_names[j]))

    edges = list(set(edges))  # remove duplicates

    print(f"\n🔗 Candidate causal edges detected: {len(edges)}")
    print(edges[:10])

except ValueError as e:
    print(f"\n❌ PC algorithm failed: {e}")
    print("Try reducing features or adding more data.")
    edges = []

# --- Step 7C: Estimate causal effects with Causal Forest (example: unit_cost → list_price) ---
if "unit_cost" in df.columns and "list_price" in df.columns:
    print("\n🌲 Running Causal Forest estimation for unit_cost → list_price...")

    Y = df["list_price"].values
    T = df["unit_cost"].values
    X = df.drop(["list_price", "unit_cost"], axis=1).values

    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X, T, Y, test_size=0.2, random_state=42
    )

    est = CausalForestDML(
        model_t=RandomForestRegressor(n_estimators=100, min_samples_leaf=10),
        model_y=RandomForestRegressor(n_estimators=100, min_samples_leaf=10),
        random_state=42,
    )

    est.fit(Y_train, T_train, X=X_train)

    treatment_effect = est.effect(X_test)
    avg_effect = np.mean(treatment_effect)

    print(f"✅ Estimated Average Causal Effect of unit_cost → list_price: {avg_effect:.4f}")

else:
    print("\n⚠️ 'unit_cost' or 'list_price' not found in dataset — check column names.")


✅ Numeric dataset ready for causal analysis
   Total_Capacity  Retail_Cost  Demand  Dealer_Cost  Manufacturing_Cost  \
0           100.0          0.0     0.0          0.0                 0.0   
1           100.0          0.0     0.0          0.0                 0.0   
2             0.0      30000.0   870.0      30000.0             30000.0   
3             0.0      30000.0  1350.0      30000.0             30000.0   
4             0.0      30000.0  1280.0      30000.0             30000.0   

   PPR1_Guide_Limit  Demand_Gap  
0               0.0         0.0  
1               0.0         0.0  
2               0.0         0.0  
3               0.0         0.0  
4               0.0         0.0  

🧹 Cleaning numeric data before causal discovery...
Condition number: 2.54e+01

🔍 Running PC Algorithm to infer causal structure...
























Depth=2, working on node 4: 100%|██████████| 5/5 [00:00<00:00, 182.97it/s]


🔗 Candidate causal edges detected: 6
[('Demand', 'Retail_Cost'), ('Retail_Cost', 'Total_Capacity'), ('PPR1_Guide_Limit', 'Retail_Cost'), ('Retail_Cost', 'PPR1_Guide_Limit'), ('Retail_Cost', 'Demand'), ('Total_Capacity', 'Retail_Cost')]

⚠️ 'unit_cost' or 'list_price' not found in dataset — check column names.


In [26]:
pip install pyvis

Note: you may need to restart the kernel to use updated packages.


In [27]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [28]:
# =====================================================
# 🌸 Bloom-style Graph Visualization from Neo4j JSON
# =====================================================

import json
import networkx as nx
from pyvis.network import Network
import random
import os

# --- Step 1: Load exported Neo4j JSON file ---
file_path = r"C:\Users\DhruvilPatel\Desktop\GAINS\neo4j_full_export.json"

with open(file_path, "r") as f:
    graph_data = json.load(f)

# --- Step 2: Build directed NetworkX graph ---
G = nx.DiGraph()

for node in graph_data["nodes"]:
    node_id = node["id"]
    labels = node.get("labels", [])
    props = node.get("props", {})

    # Friendly display label
    label_name = labels[0] if labels else "Node"
    title = "<br>".join([f"<b>{k}:</b> {v}" for k, v in props.items()])

    # Bloom-like color by label
    random.seed(label_name)
    color = "#%06x" % random.randint(0, 0xFFFFFF)

    G.add_node(
        node_id,
        label=label_name,
        title=title,
        color=color,
        group=label_name
    )

for rel in graph_data["relationships"]:
    start = rel["start_id"]
    end = rel["end_id"]
    rel_type = rel.get("type", "RELATIONSHIP")
    props = rel.get("props", {})

    title = "<br>".join([f"<b>{k}:</b> {v}" for k, v in props.items()])
    G.add_edge(start, end, label=rel_type, title=title, color="#999999")

print(f"✅ Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# --- Step 3: Create PyVis network (Bloom-style layout) ---
net = Network(height="800px", width="100%", directed=True, bgcolor="#222222", font_color="white")

# Enable physics like Bloom
net.barnes_hut()
net.from_nx(G)

# --- Step 4: Visual customization ---
net.set_options("""
var options = {
  "nodes": {
    "borderWidth": 1,
    "shape": "dot",
    "size": 15,
    "font": {"color": "white"},
    "shadow": true
  },
  "edges": {
    "color": {"inherit": false},
    "smooth": {"type": "continuous"},
    "arrows": {"to": {"enabled": true}}
  },
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -20000,
      "centralGravity": 0.4,
      "springLength": 120,
      "springConstant": 0.05,
      "avoidOverlap": 0.2
    }
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "keyboard": true,
    "tooltipDelay": 200
  }
}
""")

# --- Step 5: Export and open ---
output_file = os.path.join(os.path.dirname(file_path), "bloom_style_graph.html")
net.write_html(output_file)
print(f"🌸 Visualization ready: {output_file}")

print("\n⚙️ To open:")
print("   cd " + os.path.dirname(file_path))
print("   python -m http.server")
print("Then visit: http://localhost:8000/bloom_style_graph.html")


✅ Graph built: 334 nodes, 1128 edges
🌸 Visualization ready: C:\Users\DhruvilPatel\Desktop\GAINS\bloom_style_graph.html

⚙️ To open:
   cd C:\Users\DhruvilPatel\Desktop\GAINS
   python -m http.server
Then visit: http://localhost:8000/bloom_style_graph.html


In [29]:
import os
print("📂 Current working directory:", os.getcwd())


📂 Current working directory: c:\Users\DhruvilPatel\Desktop\GAINS
